In [111]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch.nn as nn

In [112]:
import pandas as pd

df = pd.read_csv("C:\\PROJECTS\\E Commerce Review Intelligence System\\data\\7817_1.csv")

df.sample(5)

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
802,AVpe7LD5LJeJML43ybWA,"B00DOPNO4M,B00BWYQ9YE,B00CYQPMJC,B00CUU1CGY,B0...",Amazon,"Amazon Devices,Kindle Store,buy a kindle",NaN,2015-05-22T15:33:59Z,2017-07-18T23:52:40Z,NaN,NaN,"kindlefirehdx7/b00dopno4m,kindlefirehdx7/b00bw...",...,NaN,http://www.amazon.com/kindle-fire-hdx-student-...,I love this handheld device especially all the...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,AVpfpzCi1cnluZ0-oxEr,B00DOPNLJ0,Amazon,Amazon Devices,NaN,2015-06-02T08:44:19Z,2017-08-07T15:41:59Z,NaN,NaN,kindlefirehdx89/b00dopnlj0,...,NaN,http://www.amazon.com/Kindle-Fire-HDX-Display-...,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,NaN,NaN,B. Tarbuck,NaN,NaN,NaN
350,AVpfLiCSilAPnD_xWpk_,B00CX5P8FC,Amazon,"Categories,Amazon Devices,Electronics Features...",NaN,2015-05-22T18:12:20Z,2017-08-08T22:03:26Z,NaN,8.487190e+11,"848719022827,0848719022827,amazonfiretv/b00cx5...",...,NaN,http://www.amazon.com/Fire-TV-streaming-media-...,This was easy to set up I can access many movi...,Amazon Fire TV - A must have!!!!,NaN,NaN,Amazon Customer,NaN,8.487190e+11,NaN
682,AVzRlo37glJLPUi8FbPy,B01LW1MS9C,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:46Z,NaN,NaN,amazonechodotcasefitsechodot2ndgenerationonlyc...,...,5.0,https://www.amazon.com/Amazon-Echo-Case-fits-G...,I am thoroughly impressed with my Echo Dot and...,The Extra Touch,NaN,NaN,dm,NaN,NaN,NaN
1324,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,5.0,http://reviews.bestbuy.com/3545/5097300/review...,Great little device easy to connect to bluetoo...,Easy small decent sound,NaN,NaN,Drjim,NaN,8.416670e+11,1.75 lbs


### Text preprocessing

In [113]:

df = df[['reviews.text', 'reviews.rating']]
df = df.dropna()

def label_sentiment(rating):
    if rating >= 4:
        return 1  # positive
    else:
        return 0  # negative

df['label'] = df['reviews.rating'].apply(label_sentiment)
df = df[['reviews.text', 'label']]

In [114]:
df.sample(5)

,reviews.text,label
1438,(The volume does not work with the iPhones. Pl...,1
750,This is my first kindle since the 1st generati...,1
641,I'll preface this by saying that I own an iPad...,1
420,"I got mine from an electronics store, because ...",1
1060,The Tap has been wonderful! We use it as a tim...,1


In [115]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'])

### Tokenizer

In [116]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True)

### Converting into Hugging Face dataset

In [117]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(lambda x: tokenize(x['reviews.text']), batched=True)
test_dataset = test_dataset.map(lambda x: tokenize(x['reviews.text']), batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

Map:   0%|          | 0/236 [00:00<?, ? examples/s]

### Compute Class Weights For Imabalaced Dataset

In [118]:
labels = train_df['label'].values

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = class_weights.to(device)

print("Class Weights:", class_weights)

Class Weights: tensor([2.9406, 0.6024], device='cuda:0')


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,
)
# Since here there is 2 classes positive and negative, so num_labels is 2
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Input → Model → Logits → Weighted Loss → Backpropagation
- Create a custom trainer instead of default one

- Override the default loss function

- Get the true answers (actual sentiment)

- Model makes predictions (raw scores)

- Use weighted loss
Minority class → higher penalty
Majority class → lower penalty

- Compare prediction vs actual → calculate error

- Send error back → model learns and improves

Without this:

Model ignores rare classes 

With this:

Model forced to care about them

In [120]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels") if "labels" in inputs else inputs.get("label")
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

In [132]:
# Fine-tuned model training configuration
training_args = TrainingArguments(
    
    output_dir="./results",  # Folder where model checkpoints and outputs are saved
    learning_rate=2e-5,   # Controls how fast model learns (too high = unstable, too low = slow)
    per_device_train_batch_size=16,   # Number of samples per batch during training
    per_device_eval_batch_size=32,   # Batch size for evaluation (can be higher since no backprop)
    num_train_epochs=4,  # Number of times model sees entire dataset
    weight_decay=0.1,  # Regularization to prevent overfitting (0.01–0.1 is common)
    eval_strategy="epoch",  # Evaluate model after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    load_best_model_at_end=True,  # Automatically load best model based on evaluation performance
    logging_dir="./logs"  # Directory to store training logs
)

In [122]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [123]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.494757
2,No log,0.389537
3,No log,0.419098
4,No log,0.395089


TrainOutput(global_step=236, training_loss=0.35774848420741195, metrics={'train_runtime': 114.0383, 'train_samples_per_second': 33.006, 'train_steps_per_second': 2.069, 'total_flos': 498607288541184.0, 'train_loss': 0.35774848420741195, 'epoch': 4.0})

In [124]:
trainer.evaluate()

{'eval_loss': 0.3895372152328491,
 'eval_runtime': 3.3111,
 'eval_samples_per_second': 71.276,
 'eval_steps_per_second': 2.416,
 'epoch': 4.0}

In [125]:
# Predictions
train_preds = trainer.predict(train_dataset)
test_preds = trainer.predict(test_dataset)

# Convert logits → labels
train_pred_labels = np.argmax(train_preds.predictions, axis=1)
test_pred_labels = np.argmax(test_preds.predictions, axis=1)

train_true = train_preds.label_ids
test_true = test_preds.label_ids

labels_names = ["negative", "positive"]

print("🔹 TRAIN REPORT")
print(classification_report(train_true, train_pred_labels, target_names=labels_names))

print("\n🔹 TEST REPORT")
print(classification_report(test_true, test_pred_labels, target_names=labels_names))

🔹 TRAIN REPORT
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86       160
    positive       0.98      0.97      0.97       781

    accuracy                           0.95       941
   macro avg       0.91      0.92      0.92       941
weighted avg       0.95      0.95      0.95       941


🔹 TEST REPORT
              precision    recall  f1-score   support

    negative       0.74      0.78      0.76        40
    positive       0.95      0.94      0.95       196

    accuracy                           0.92       236
   macro avg       0.85      0.86      0.85       236
weighted avg       0.92      0.92      0.92       236



In [133]:
def predict(text):
    model.eval() #Set model to evaluation mode (no training, no dropout)
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()} #Move data to same device (CPU/GPU) as model
    
    with torch.no_grad():
        outputs = model(**inputs) # Run model without calculating gradients (faster + efficient)
    
    logits = outputs.logits  #Get raw prediction scores 
    pred = torch.argmax(logits, dim=1).item() #Pick the highest score → predicted class index
    
    labels = ["negative", "positive"]
    return labels[pred]

### Converts text → model prediction → returns sentiment label

In [127]:

print(predict("This product is amazing but delivery was slow"))

negative


In [128]:
print(predict("This is the worst product I have ever bought"))

negative


In [129]:
print(predict("The product is okay, nothing special"))

positive


In [130]:
print(predict("The product is good but delivery was very slow"))

negative


### For Fine tunninig the model
1. Tune Learning Rate
2. Increase Epochs
3. Batch Size Tuning
4. Use Better Model
5. Add Weight Decay (Regularization)
6. Warmup Steps (Advanced but useful)
7. Evaluation Strategy + Early Stopping

training_args = TrainingArguments(
   
    output_dir="./results",
   
    learning_rate=2e-5,
   
    per_device_train_batch_size=16,
   
    per_device_eval_batch_size=16,
   
    num_train_epochs=4,
   
    weight_decay=0.01,
   
    evaluation_strategy="epoch",
   
    save_strategy="epoch",
   
    load_best_model_at_end=True,
   
    logging_dir="./logs"
)

## Save The model

In [134]:
# Save model
model.save_pretrained("sentiment_model")

# Save tokenizer
tokenizer.save_pretrained("sentiment_model")

('sentiment_model\\tokenizer_config.json',
 'sentiment_model\\special_tokens_map.json',
 'sentiment_model\\vocab.txt',
 'sentiment_model\\added_tokens.json')